# 02 · pandas
NumPy helped us calculate with arrays. pandas adds **row and column labels**
and tools for working with tables whose columns can have different types.
Our workflow is **inspect → select → summarize → visualize**.

We will start with three illustrative flowers, then explore the Iris dataset:
150 flowers, four measurements in centimetres, and three species. Iris comes
with scikit-learn; `load_iris` does not download data. The next notebook uses
two of these species for classification.

**How to use this notebook:** predict a result, run the cell, then explain
what one row of the result represents. `display(...)` shows a formatted table
in Jupyter or Colab. Plotting code is provided; focus on the table behind each
graph. Run from top to bottom with NumPy, pandas, Matplotlib and scikit-learn
installed. No extra data files are needed.

**In class:** follow sections A–E, including Exercises E and F and the
summary tables, bar chart and scatter plot.
**After class:** the **Exercises for you** section is for practice at your own pace.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from IPython.display import display

# Display precision only; stored values keep their full precision.
pd.set_option("display.precision", 3)

## A. Build a table you can read
A **DataFrame** is a 2D table. Its columns have names, and its rows have an
**index**. A dictionary of equally sized lists is one way to create it:
dictionary keys become column names.

These three flowers are illustrative values. The row labels are identifiers,
not measurements. Notice that `species` contains text while the other
columns contain numbers.

In [ ]:
mini = pd.DataFrame({
    "species": ["setosa", "versicolor", "virginica"],
    "petal_length_cm": [1.4, 4.7, 6.0],
    "petal_width_cm": [0.2, 1.4, 2.5],
}, index=["flower_a", "flower_b", "flower_c"])
display(mini)

### One column: Series or DataFrame?
A **Series** is a 1D labelled column. A list of column names keeps a 2D
DataFrame, even when that list contains only one name.

| Expression | Result | Shape |
|---|---|---|
| `mini["petal_length_cm"]` | Series | `(3,)` |
| `mini[["petal_length_cm"]]` | DataFrame | `(3, 1)` |

This is similar to keeping or removing a dimension in NumPy, but the row
labels stay attached to the values.

In [ ]:
lengths = mini["petal_length_cm"]
length_table = mini[["petal_length_cm"]]
print("Series shape:", lengths.shape)
display(lengths)
print("DataFrame shape:", length_table.shape)
display(length_table)

## B. Inspect before calculating
Load the full Iris table and translate the numeric `target` codes into
species names. The numeric code is a **category**, not a measurement to
average. We will explicitly select measurement columns for statistics.

`head()` shows the first five rows. Here those rows are all setosa, so they
do not show the full variety in the dataset.

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()

# Iris uses three numeric codes. This dictionary gives the name for each code.
species_names = {0: "setosa", 1: "versicolor", 2: "virginica"}
# map looks up each target code in the dictionary to create a column of names.
df["species"] = df["target"].map(species_names)
display(df.head())

| Tool | Question it answers |
|---|---|
| `df.shape` | How many rows and columns? |
| `df.info()` | What are the column types and non-missing counts? |
| `df.describe()` | What are the numeric ranges and summaries? |
| `df.isna().sum()` | How many missing entries are in each column? |

**Read the output:** expect 150 rows and six columns: four measurements,
a target code, and a species name. In `describe`, `50%` is the median;
`25%` and `75%` delimit the middle half of the values. `.round(2)` below
rounds the displayed summary, not the measurements in `df`.

In [ ]:
print("Shape:", df.shape)

df.info()

In [ ]:
measurement_columns = iris.feature_names
print("Measurement columns:", measurement_columns)
measurements = df[measurement_columns]
# First compute the statistics, then round the summary for display.
measurement_summary = measurements.describe()
rounded_summary = measurement_summary.round(2)
display(rounded_summary)

## C. Select rows and columns
Both selectors use **`[rows, columns]`**, like the NumPy examples.

| Selector | Uses | Slice endpoint |
|---|---|---|
| `.loc` | Row/column labels, or a Boolean mask | Includes the stop label |
| `.iloc` | Integer positions | Excludes the stop position |

Compare the two results below. On this ordered index, both select the first
two flowers and the same two columns. `flower_b` is a label; position `1`
is where that row happens to sit. Sorting can change positions without
changing labels.

In [ ]:
# reminder mini
print("Mini DataFrame:")
display(mini)

print("Selection:")
display(mini.loc["flower_a":"flower_b", ["species", "petal_width_cm"]])
display(mini.iloc[:2, [0, 2]])

### Turn a question into a mask
Which Iris flowers have petal length above `5` cm? The comparison produces
one Boolean value per row. `.loc` uses that mask to keep whole rows and the
named columns. `sort_values` then puts the longest petals first.

To combine conditions, use `&` for **and** or `|` for **or**, putting each
comparison in parentheses. Use `.isna()` to test missingness.

In [ ]:
long_mask = df["petal length (cm)"] > 5
selected = df.loc[long_mask, ["species", "petal length (cm)"]]
ranked = selected.sort_values("petal length (cm)", ascending=False)
print("Rows selected:", len(selected))
display(ranked.head())

## D. From individual rows to a summary table
`value_counts()` counts rows in each category. `groupby("species")` splits
the flowers by species; selecting two measurement columns and calling
`.mean()` computes one mean per species and feature.

**Predict:** how many rows and measurement columns will `petal_means` have?
A row in `df` is a flower; a row in `petal_means` is a species summary.

In [ ]:
# Count the flowers in each species; the species names become the index.
species_counts = df["species"].value_counts()

# Sort that index alphabetically, then turn the Series into a one-column table.
species_counts = species_counts.sort_index()
species_counts = species_counts.to_frame(name="flowers")

# Group the rows, select the measurements, then compute each group’s means.
petal_columns = ["petal length (cm)", "petal width (cm)"]
species_groups = df.groupby("species")
grouped_petals = species_groups[petal_columns]
petal_means = grouped_petals.mean()

display(species_counts)
display(petal_means.round(2))

### The same summary, shown as bars
pandas can plot directly from a table: the index provides the category
labels, and the numeric columns provide the bar heights.

**Read it:** each pair of bars comes from one row of `petal_means`. Which
species has the longest petals on average? Can these means tell us how
much the individual flowers overlap?

In [ ]:
ax = petal_means.plot.bar(figsize=(6.5, 3.5), rot=0,
                         color=["#0072B2", "#D55E00"])
ax.set(xlabel="Species", ylabel="Mean measurement (cm)",
       title="One summary row becomes a pair of bars")
ax.legend(["Petal length", "Petal width"])
plt.tight_layout()
plt.show()

### Exercise E · Selection and aggregation (6 min)
1. Select the virginica flowers with petal width above `2.0` cm.
2. Compute their mean petal length.
3. Produce the mean petal length and width for **each species**, using `groupby`.

**Checkpoints:** 23 selected rows; mean length about `5.76087` cm; the final
table has three rows and two measurement columns. Show the filtered table's
first few rows so you can check both conditions.

<details>
<summary>Hint</summary>

Start with `(df["species"] == "virginica") & (df["petal width (cm)"] > 2.0)`.
Select with `.loc`, then take the mean of the selected petal lengths.
Step 3 uses the full `df`, not only the filtered flowers.

</details>

In [ ]:
# 1. Build the two-condition mask and display the selected rows.

# 2. Compute their mean petal length.

# 3. Summarize both petal measurements for each species in the full df.

## E. Missing values and individual observations
Iris has no missing measurements. To make missingness visible, insert one
into a **copy**. `.loc[row_label, column_label] = value` updates that cell.
`np.nan` means a numeric value is missing; it does not mean zero.

`isna()` produces a Boolean table, and `.sum()` counts `True` values per
column. `dropna(subset=[...])` removes rows missing the specified measurement.
**Predict:** how many rows will remain? Will the original `df` change?

In [ ]:
messy = df.copy()
messy.loc[0, "sepal width (cm)"] = np.nan

# isna marks missing cells True; sum counts those True values in each column.
original_missing = df.isna()
edited_missing = messy.isna()
original_missing_counts = original_missing.sum()
edited_missing_counts = edited_missing.sum()

missing_counts = pd.DataFrame({
    "original": original_missing_counts,
    "after edit": edited_missing_counts,
})
display(missing_counts)
missing_width_mask = edited_missing["sepal width (cm)"]
missing_rows = messy.loc[missing_width_mask]
display(missing_rows)

clean = messy.dropna(subset=["sepal width (cm)"])
print("Rows before / after:", len(messy), len(clean))
assert len(clean) == len(df) - 1
# Check that every sepal width in the original table is still present.
original_widths_present = df["sepal width (cm)"].notna()
assert original_widths_present.all()

Dropping is a choice for this exercise. In practice, investigate why a value
is missing before deciding to remove or fill it. Numeric means skip missing
values by default: `.count()` counts observed values, while `len(...)` counts
rows. Filling with zero would change the measurement and its mean.

Dropping rows preserves the remaining index labels. For a model, any learned
imputation belongs inside the training process.

### Exercise F · A plot and a sentence (5 min)
Run the scatter plot. Each point represents one flower, so it complements
the means in the bar chart.

1. Describe one pattern and one limitation.
2. Change `x_feature` to `"sepal length (cm)"`, rerun, and compare. The
   horizontal label follows this variable automatically.

**Checkpoint:** the legend identifies species and both axes state units.
A visible association does not establish causation or performance on future data.

In [ ]:
x_feature = "petal length (cm)"
y_feature = "petal width (cm)"
# Provided styles distinguish species with both colour and marker shape.
styles = {"setosa": ("o", "#0072B2"),
          "versicolor": ("s", "#D55E00"),
          "virginica": ("^", "#009E73")}
fig, ax = plt.subplots(figsize=(6.5, 3.8))
# Each loop step gives a species name and the table of flowers in that species.
for species, group in df.groupby("species"):
    # Unpack the two style values: marker shape first, colour second.
    marker, color = styles[species]
    ax.scatter(group[x_feature], group[y_feature], label=species,
               marker=marker, color=color, alpha=0.75)
ax.set(xlabel=x_feature, ylabel=y_feature,
       title="Individual flowers show variation within each species")
ax.legend()
fig.tight_layout()
plt.show()

**Your observations:**

- Pattern: …
- Limitation: …
- After changing the horizontal feature: …

### Pause and explain
How is a Series different from a DataFrame? What do `.loc` and `.iloc` use
to find rows? What does a row of a grouped summary represent? What can the
scatter plot show that a mean cannot?

## Exercises for you

These exercises and explorations are for you to work through after class.
Use the tables prepared above, modify the provided examples, and explain
what you observe. Hints and reference solutions are at the end of the notebook.
Do exercise 3 before exercise 4: the duplicate-record example reuses `sales`.

### 1. A frequency table and a histogram
We used `value_counts` for species categories. For a numeric measurement
with many distinct values, a histogram groups values into **intervals**
called bins. It counts observations, not species averages.

**Try it:** change `bins` from `12` to `6`. Does the number of observations
change? The graph pools all three species; what information does that hide?

In [ ]:
display(species_counts)
petal_lengths = df["petal length (cm)"]
print("Petal lengths included:", petal_lengths.count())
ax = petal_lengths.plot.hist(
    bins=12, figsize=(6.5, 3.2), color="#0072B2", edgecolor="white")
ax.set(xlabel="Petal length (cm)", ylabel="Number of flowers",
       title="150 measurements grouped into intervals")
plt.tight_layout()
plt.show()

### 2. Update a table and try three small puzzles
Column arithmetic works on all rows, as in NumPy. Work on a copy, add a
derived column, then give it a clearer name. `.rename` and `.drop` return
new tables by default, so assign the result when you want to keep it.

The new millimetre column contains the same information in different units.

In [ ]:
enriched = mini.copy()
enriched["length_mm"] = enriched["petal_length_cm"] * 10
enriched = enriched.rename(columns={"length_mm": "petal_length_mm"})
display(enriched)
without_extra = enriched.drop(columns=["petal_length_mm"])
print("After dropping the extra column:", without_extra.shape)

These short tasks echo the **DataFrame basics** section of pandas puzzles
(selection, missing values and updating a labelled row). Use the practice
copy below; the missing value was deliberately inserted.

1. Display just the flower whose width is missing.
2. Display all three rows sorted by petal length, largest first.
3. Our source record confirms the missing width is `1.4` cm. Restore that
   one value using `.loc` and check that no widths are missing.

**Checkpoints:** row `flower_b`; order `flower_c`, `flower_b`, `flower_a`;
zero missing widths after the correction. Sorting does not rename row labels.

In [ ]:
practice = mini.copy()
practice.loc["flower_b", "petal_width_cm"] = np.nan
display(practice)
# Find the missing row, sort by length, then restore the known width.

### 3. Read a small sales table and pivot it
Explore CSV input, grouping and pivoting with six **invented orders**,
small enough to check by hand.
Each row is one order, and `quantity` counts items.

`read_csv` reads comma-separated data. `StringIO` makes the text below act
like a file, so the example works without downloading anything. With an
actual file, the same operation would be `pd.read_csv("sales.csv")`.

In [ ]:
from io import StringIO

csv_text = """order_id,country,product,quantity
101,Canada,Notebook,4
102,Canada,Pen,10
103,Canada,Notebook,6
104,France,Notebook,3
105,France,Pen,8
106,France,Pen,2
"""
# Wrap the text as a file-like object, then read it as a CSV table.
csv_file = StringIO(csv_text)
sales = pd.read_csv(csv_file, skipinitialspace=True)
display(sales)

#### Six order rows → four totals → a comparison table
First group by country **and** product and sum quantities. `as_index=False`
keeps the grouping labels as ordinary columns in this summary.

A pivot puts one label on the rows and another on the columns. Plain
`pivot` requires one value per row/column pair. Our raw orders repeat some
country/product pairs, so `pivot_table(..., aggfunc="sum")` combines them.
Summing answers **how many items were ordered**, not how many orders there were.

**Predict:** Canada's notebook total is `4 + 6 = 10`. Fill in the other three
totals before running. Every country/product pair occurs in this example.

In [ ]:
# Make one group for each country/product pair, then sum its quantities.
order_groups = sales.groupby(["country", "product"], as_index=False)
grouped_quantities = order_groups["quantity"]
order_totals = grouped_quantities.sum()

by_country = sales.pivot_table(index="country", columns="product",
                               values="quantity", aggfunc="sum")
display(order_totals)
display(by_country)
assert by_country.loc["Canada", "Notebook"] == 10

In [ ]:
ax = by_country.plot.bar(figsize=(6, 3.2), rot=0,
                        color=["#0072B2", "#D55E00"])
ax.set(xlabel="Country", ylabel="Items ordered",
       title="A pivot table becomes a grouped bar chart")
ax.legend(title="Product")
plt.tight_layout()
plt.show()

**Try it:** make a new pivot with products on the rows and countries on the
columns, keeping `aggfunc="sum"`. Then plot that table. Which labels move
into the legend? **Checkpoint:** the total across all cells stays `33` items.

In [ ]:
# Build a product-by-country pivot table and plot it.

### 4. Notice a duplicated record
`pd.concat` combines tables. Here we deliberately append an exact copy of
the first order; `ignore_index=True` gives the combined table a fresh row index.

`duplicated()` flags later copies of a row by default. `drop_duplicates()`
keeps the first copy. We know this extra row repeats the same order because
we just copied it, including its `order_id`. Two different orders can have
the same quantities, so do not treat matching measurements alone as an error.

**Predict, then check:** how many rows and items will there be before and
after removing the duplicate? Explain why the total quantity changes.

In [ ]:
# The list [0] keeps the first row as a DataFrame, ready to append.
first_order = sales.iloc[[0]]
repeated = pd.concat([sales, first_order], ignore_index=True)

duplicate_mask = repeated.duplicated()
duplicate_rows = repeated.loc[duplicate_mask]
display(duplicate_rows)
deduplicated = repeated.drop_duplicates()
print("Rows before / after:", len(repeated), len(deduplicated))
total_before = repeated["quantity"].sum()
total_after = deduplicated["quantity"].sum()
print("Total quantity before / after:", total_before, total_after)
assert len(deduplicated) == 6
assert total_after == 33

## Check your work
Try the tasks before opening the solutions. Unfinished practice cells do
not feed the worked examples, so the notebook can still run from the top.

<details>
<summary>Exercise E · Reference solution</summary>

```python
virginica_mask = df["species"] == "virginica"
wide_petal_mask = df["petal width (cm)"] > 2.0
# & keeps rows where both conditions are True.
exercise_mask = virginica_mask & wide_petal_mask
wide_virginica = df.loc[exercise_mask]
mean_length = wide_virginica["petal length (cm)"].mean()
exercise_groups = df.groupby("species")
exercise_petals = exercise_groups[petal_columns]
exercise_means = exercise_petals.mean()
display(wide_virginica.head())
print("Rows:", len(wide_virginica), "Mean length:", mean_length)
display(exercise_means)
assert len(wide_virginica) == 23
assert np.isclose(mean_length, 5.760869565217392)
assert exercise_means.shape == (3, 2)
```

</details>

<details>
<summary>Exercise F and histogram · Example observations</summary>

In the petal scatter plot, setosa occupies the lower-left region, while
versicolor and virginica overlap. Flowers with longer petals also tend to
have wider petals in this dataset. The plot alone does not measure how
well a model would predict unseen flowers.

With sepal length on the horizontal axis, the species overlap more along
that axis. Changing a histogram's bins redraws the same 150 values in
different intervals; it does not change the sample or its mean. Pooling
species in a histogram hides which species contributed each observation.

</details>

<details>
<summary>Small table puzzles · Reference solution</summary>

```python
practice = mini.copy()
practice.loc["flower_b", "petal_width_cm"] = np.nan
missing_width_mask = practice["petal_width_cm"].isna()
missing_flower = practice.loc[missing_width_mask]
longest_first = practice.sort_values("petal_length_cm", ascending=False)
display(missing_flower)
display(longest_first)
practice.loc["flower_b", "petal_width_cm"] = 1.4
display(practice)
assert missing_flower.index.tolist() == ["flower_b"]
assert longest_first.index.tolist() == ["flower_c", "flower_b", "flower_a"]
missing_after = practice["petal_width_cm"].isna()
assert missing_after.sum() == 0
```

</details>

<details>
<summary>Pivot challenge · Reference solution</summary>

```python
by_product = sales.pivot_table(index="product", columns="country",
                               values="quantity", aggfunc="sum")
display(by_product)
ax = by_product.plot.bar(rot=0, figsize=(6, 3.2),
                        color=["#0072B2", "#D55E00"])
ax.set(xlabel="Product", ylabel="Items ordered", title="The same totals, rearranged")
ax.legend(title="Country")
plt.tight_layout()
plt.show()
# First sum each country column, then add the country totals.
country_totals = by_product.sum()
grand_total = country_totals.sum()
assert grand_total == 33
```

Canada and France now appear in the legend. The four totals are still
`10`, `10`, `3`, and `10`; rearranging the table does not change the data.

</details>

## Sources and further practice

- [100 pandas puzzles by ajcr](https://github.com/ajcr/100-pandas-puzzles):
  light inspiration from the **DataFrame basics** questions, especially
  **4–11, 13 and 15–17**. The tiny-table questions here use our own flower
  data and wording. The main lesson stays at an introductory level.


Reference: [Iris dataset and provenance](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html),
[10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html),
[missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html),
[pivot tables](https://pandas.pydata.org/docs/user_guide/reshaping.html), and
[plotting from pandas](https://pandas.pydata.org/docs/user_guide/visualization.html).
Reading: *Python Data Science Handbook*, Data Manipulation with Pandas and
Visualization with Matplotlib.